# Phase split parity verification

Confirms that the new decomposed historical pipeline reproduces both `observed_flow` and `observed_inventory` exactly, and produces the same inventory trajectory as the original `OrganicDeparturePhase + OrganicArrivalPhase`.

Pipeline under test:

1. `HistoricalLatentDemandPhase` — publishes O_i, D_j marginals
2. `HistoricalODStructurePhase` — publishes P(j | i)
3. `DeparturePhysicsPhase(mode="permissive")` — subtracts O_i from inventory
4. `HistoricalTripSamplingPhase` — replays observed trips into in_transit
5. `ArrivalsPhase` — delivers in_transit, emits flow events

Each subsequent cell ends with a `*_mismatches` DataFrame; for parity to hold, all three must be empty.

In [1]:
# %% 2. Imports
import sys
from pathlib import Path

for _candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (_candidate / "gbp").is_dir() and (_candidate / "tests").is_dir():
        sys.path.insert(0, str(_candidate))
        break

from gbp.build.pipeline import build_model
from gbp.consumers.simulator import (
    ArrivalsPhase,
    DeparturePhysicsPhase,
    Environment,
    EnvironmentConfig,
    HistoricalLatentDemandPhase,
    HistoricalODStructurePhase,
    HistoricalTripSamplingPhase,
    OrganicArrivalPhase,
    OrganicDeparturePhase,
)
from gbp.loaders import DataLoaderGraph, DataLoaderMock, GraphLoaderConfig

simulator_imports_loaded = True

In [2]:
# %% 3. Build resolved mock model
import dataclasses

mock_config = {
    "n_stations": 10,
    "n_depots": 2,
    "n_timestamps": 168,
    "time_freq": "h",
    "start_date": "2025-01-01",
    "num_resources": 3,
    "resource_capacity": 100,
    "ebike_fraction": 0.3,
    "depot_capacity": 200,
    "seed": 42,
}
mock = DataLoaderMock(mock_config)
loader = DataLoaderGraph(mock, GraphLoaderConfig())
raw = loader.load()
resolved_full = build_model(raw)

# DataLoaderGraph derives `supply` from the same trip events that observed_flow
# already encodes on its target side.  Pre-populating in_transit from supply
# would make ArrivalsPhase deliver inflow that HistoricalTripSampling also
# produces, double-counting arrivals.  For an apples-to-apples replay we
# drop supply; observed_flow remains the single source of trip events.
resolved = dataclasses.replace(resolved_full, supply=None)

2026-04-30 21:33:43 [info     ] load_start                     loader=graph_core
2026-04-30 21:33:45 [debug    ] source_validated               loader=graph_core
2026-04-30 21:33:45 [debug    ] observed_flow_built            loader=graph_core rows=727
2026-04-30 21:33:45 [debug    ] observed_inventory_built       loader=graph_core rows=140
2026-04-30 21:33:45 [info     ] load_done                      facilities=12 loader=graph_core


In [3]:
# %% 4. Reference run: OrganicDeparture + OrganicArrival
reference_log = Environment(
    resolved,
    EnvironmentConfig(
        phases=[OrganicDeparturePhase(), OrganicArrivalPhase()],
        seed=42,
        scenario_id="reference",
    ),
).run()
reference_tables = reference_log.to_dataframes()

In [4]:
# %% 5. Decomposed run: full historical phase split
decomposed_log = Environment(
    resolved,
    EnvironmentConfig(
        phases=[
            HistoricalLatentDemandPhase(),
            HistoricalODStructurePhase(),
            DeparturePhysicsPhase(mode="permissive"),
            HistoricalTripSamplingPhase(),
            ArrivalsPhase(),
        ],
        seed=42,
        scenario_id="decomposed",
    ),
).run()
decomposed_tables = decomposed_log.to_dataframes()

In [5]:
# %% 6. Flow parity: decomposed simulation_flow_log vs resolved.observed_flow
flow_keys = ["period_id", "source_id", "target_id", "commodity_category"]
decomposed_flow = (
    decomposed_tables["simulation_flow_log"]
    .groupby(flow_keys, as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "simulated_quantity"})
)
observed_flow = (
    resolved.observed_flow
    .groupby(flow_keys, as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "observed_quantity"})
)
flow_compare = decomposed_flow.merge(observed_flow, on=flow_keys, how="outer")
flow_compare["simulated_quantity"] = flow_compare["simulated_quantity"].fillna(0.0)
flow_compare["observed_quantity"] = flow_compare["observed_quantity"].fillna(0.0)
flow_compare["diff"] = (
    flow_compare["simulated_quantity"] - flow_compare["observed_quantity"]
)
flow_mismatches = flow_compare.loc[
    flow_compare["diff"].abs() > 1e-9
].reset_index(drop=True)
flow_mismatches

,period_id,source_id,target_id,commodity_category,simulated_quantity,observed_quantity,diff


In [6]:
# %% 7. Inventory parity: decomposed simulation_inventory_log vs resolved.observed_inventory
inventory_keys = ["period_id", "facility_id", "commodity_category"]
observed_facility_ids = resolved.observed_inventory["facility_id"].unique()
decomposed_inventory = (
    decomposed_tables["simulation_inventory_log"]
    .loc[lambda d: d["facility_id"].isin(observed_facility_ids)]
    .groupby(inventory_keys, as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "simulated_quantity"})
)
observed_inventory = (
    resolved.observed_inventory
    .groupby(inventory_keys, as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "observed_quantity"})
)
inventory_compare = decomposed_inventory.merge(
    observed_inventory, on=inventory_keys, how="outer",
)
inventory_compare["simulated_quantity"] = (
    inventory_compare["simulated_quantity"].fillna(0.0)
)
inventory_compare["observed_quantity"] = (
    inventory_compare["observed_quantity"].fillna(0.0)
)
inventory_compare["diff"] = (
    inventory_compare["simulated_quantity"]
    - inventory_compare["observed_quantity"]
)
inventory_mismatches = inventory_compare.loc[
    inventory_compare["diff"].abs() > 1e-9
].reset_index(drop=True)
inventory_mismatches

,period_id,facility_id,commodity_category,simulated_quantity,observed_quantity,diff


In [7]:
# %% 8. Pipeline equivalence: decomposed inventory vs reference inventory
reference_inventory_full = (
    reference_tables["simulation_inventory_log"]
    .groupby(inventory_keys, as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "reference_quantity"})
)
decomposed_inventory_full = (
    decomposed_tables["simulation_inventory_log"]
    .groupby(inventory_keys, as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "decomposed_quantity"})
)
pipeline_compare = reference_inventory_full.merge(
    decomposed_inventory_full, on=inventory_keys, how="outer",
)
pipeline_compare["reference_quantity"] = (
    pipeline_compare["reference_quantity"].fillna(0.0)
)
pipeline_compare["decomposed_quantity"] = (
    pipeline_compare["decomposed_quantity"].fillna(0.0)
)
pipeline_compare["diff"] = (
    pipeline_compare["decomposed_quantity"]
    - pipeline_compare["reference_quantity"]
)
pipeline_mismatches = pipeline_compare.loc[
    pipeline_compare["diff"].abs() > 1e-9
].reset_index(drop=True)
pipeline_mismatches

,period_id,facility_id,commodity_category,reference_quantity,decomposed_quantity,diff
